# Forward information and incremental information beyond FFT

**IC** relates a causal feature at *t* to an outcome over *t+1 .. t+h*. **Models A-D** are nested linear / logistic research models fitted on chronological, expanding training sets with an embargo: A statistical, B + OU, C + Fourier, D + wavelet. The question is only whether D beats C out of sample - and by more than the same models manage on the null controls.

**This notebook contains no calculations.** Everything is computed by `xauusd_quant.features.wavelet*` and `xauusd_quant.research.wavelet_*` and read back from `results/wavelet_research/<tf>/<series>/window_<N>/`. Generate those first:

```
xq wavelet-research --timeframe 5m --source regression_residual --window 512
xq wavelet-research --timeframe 5m --all-windows
xq wavelet-research --all
xq wavelet-research --controls
```

**Causal vs offline.** Per-bar features are *causal*: a one-sided MODWT on the trailing window, identical whether or not later bars exist, and the only thing ever written to `data/features/wavelet/`. Scalograms and ridges are computed from whole slices, centred on every point, and are labelled **NON-CAUSAL - VISUALIZATION ONLY**; they never become features.

Every number sits beside the null controls: white noise, a detrended random walk (the same regression and OU pipeline on a series with no structure), shuffled returns and block-bootstrapped returns (blocks of 64, 256 and 1024 bars). No trading rule is defined or evaluated anywhere. No costs are applied. Nothing here is a signal.

In [ ]:
import sys
import json
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl
from IPython.display import Image, display

from xauusd_quant.features.wavelet_config import load_wavelet_config

wavelet = load_wavelet_config(ROOT / "config" / "wavelet.yaml")

TIMEFRAME = "5m"                    # change and re-run
SERIES = "regression_residual"      # regression_residual | ou_innovation | log_return
WINDOW = 512                        # rolling window N (bars)

results = wavelet.results_dir(TIMEFRAME, SERIES, WINDOW)
if not results.exists():
    raise SystemExit(
        f"No results for {TIMEFRAME} {SERIES} N={WINDOW}. Run:  xq wavelet-research "
        f"--timeframe {TIMEFRAME} --source {SERIES} --window {WINDOW}")

def table(name):
    """Load a previously generated wavelet table (None if it was not written)."""
    path = results / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

def plot(name, folder=None):
    path = (folder or results / "plots") / f"{name}.png"
    if path.exists():
        display(Image(filename=str(path)))

payload = json.loads((results / "summary.json").read_text(encoding="utf-8"))
summary = payload["summary"]
lineage = payload["provenance"]["dataset"]
print(f"bars       : {summary['observations']:,} ({summary['valid_feature_fraction']:.1%} "
      f"with a valid {WINDOW}-bar window)")
print(f"ticks      : {lineage['tick_dataset_version']} ({lineage['tick_dataset_status']})")
print(f"bars       : {lineage.get('bar_dataset_version')}")
print(f"code       : {payload['provenance']['code_fingerprint']}, "
      f"PyWavelets {payload['provenance']['packages'].get('PyWavelets')}")
if lineage.get("label"):
    print(f"WARNING    : {lineage['label']}")
for warning in payload.get("warnings", []):
    print(f"WARNING    : {warning}")

## Information coefficients

In [ ]:
ic = table('ic_analysis')
print("tests per source:", summary["ic_tests"])
ic.sort(pl.col("rank_ic").abs(), descending=True).head(20)

In [ ]:
plot('ic_by_horizon')

## Residual and OU extremes by wavelet state (and by OU state, for comparison)

In [ ]:
table('extreme_conditioning').filter(pl.col("threshold") == 2.0)

## Incremental information: models A-D

In [ ]:
table('incremental_summary')

In [ ]:
table('incremental_by_feature').group_by("model", "target", "horizon").agg(
    pl.col("oos_r2").mean()).sort("model", "target", "horizon")

## Feature quality (Step 48)

In [ ]:
table('feature_quality')